# Python for MATLAB users · 3 · Arrays and numbers

NumPy arrays: views, products, broadcasting, arithmetic, memory order, logic.

**Run this cell first** (Shift+Enter). It downloads the project and installs it
into the Python of this notebook. Every notebook runs on its own machine, so every
notebook starts with this cell, and needs it again when Colab gives it a new machine.

In [ ]:
!test -d /content/pdm-workshop || git clone -q --depth 1 https://github.com/alessandro-rubin-sate/pdm-workshop.git /content/pdm-workshop
%cd /content/pdm-workshop
%pip install -q .

## Where the language bites

On the trap cells: **read the cell and say what it will print before you run it.**
Then run it, and name the habit that prevents the mistake.

### Trap 3 - Slices are views; masks are copies.

`traps/03_views.py`

Predict the content of `a` after each block.

In [ ]:
import numpy as np

In [ ]:
# A slice shares memory with the array it comes from
a = np.zeros(5)
window = a[1:4]
window[:] = 9.0
print("after writing through the slice:", a)
print("same memory:", np.shares_memory(a, window))

In [ ]:
# Indexing with a mask (or a list of indices) returns a copy
selected = a[a > 0]
selected[:] = 5.0
print("after writing through the mask: ", a)
print("same memory:", np.shares_memory(a, selected))

In [ ]:
# The habit: copy the slice you are going to modify
window = a[1:4].copy()
window[:] = 0.0
print("after writing through a copy:   ", a)

### Trap 8 - 1-D arrays and two product operators.

`traps/08_arrays_and_products.py`

```
MATLAB:  A * B is the matrix product, A .* B is element by element.
Python:  A @ B is the matrix product, A * B is element by element.
```

In [ ]:
import numpy as np

In [ ]:
# A vector has one dimension: neither a row nor a column
v = np.array([1.0, 2.0, 3.0])
print("v.shape:  ", v.shape)
print("v.T.shape:", v.T.shape)

In [ ]:
# Two operators
A = np.ones((2, 3))
print("A * 2:\n", A * 2)
print("A @ v:", A @ v, "shape", (A @ v).shape)
print("A * v:\n", A * v)

In [ ]:
# len is the size of the first dimension, not the largest one
print("len(A):", len(A), "| A.shape:", A.shape, "| A.size:", A.size)

### Trap 9 - Broadcasting can build a matrix by accident.

`traps/09_broadcasting.py`

The predictions are identical to the targets, so the squared error must be 0.
Predict the shape of `error` and the value printed.

In [ ]:
import numpy as np

In [ ]:
# A column minus a 1-D array
target = np.array([1.0, 2.0, 3.0])
prediction = target.reshape(3, 1)  # what many models return: one column
error = target - prediction
print("target.shape:    ", target.shape)
print("prediction.shape:", prediction.shape)
print("error.shape:     ", error.shape)
print("mean squared error:", (error**2).mean())

In [ ]:
# The habit: check the shapes, flatten before comparing
error = target - prediction.ravel()
print("error.shape:     ", error.shape)
print("mean squared error:", (error**2).mean())

### Trap 10 - Arithmetic that looks the same and is not.

`traps/10_arithmetic.py`

For each line, write down what MATLAB gives, then what you expect from Python.

In [ ]:
import numpy as np

In [ ]:
# Power: MATLAB 2^3 is 8
print("2 ^ 3: ", 2 ^ 3)
print("2 ** 3:", 2**3)

In [ ]:
# Rounding: MATLAB round(2.5) is 3
print("round(2.5):", round(2.5))
print("round(3.5):", round(3.5))
print("np.round([0.5, 1.5, 2.5]):", np.round([0.5, 1.5, 2.5]))

In [ ]:
# An integer array stays integer: MATLAB a = [1 2 3]; a(1) = 0.7 keeps 0.7
a = np.array([1, 2, 3])
a[0] = 0.7
print("a:", a, "dtype:", a.dtype)

b = np.array([1.0, 2, 3])
b[0] = 0.7
print("b:", b, "dtype:", b.dtype)

In [ ]:
# Integers wrap around: MATLAB uint8(200) + uint8(100) is 255
counts = np.array([200], dtype=np.uint8)
print("200 + 100 as uint8:", counts + np.array([100], dtype=np.uint8))

In [ ]:
# Division always gives a float, and a float is not an index
x = np.arange(10)
n = len(x)
print("n / 2: ", n / 2)
print("n // 2:", n // 2)
try:
    print(x[n / 2])
except IndexError as error:
    print("x[n / 2]: IndexError:", error)
print("x[n // 2]:", x[n // 2])

### Trap 11 - MATLAB fills columns first, NumPy fills rows.

`traps/11_memory_order.py`

```
MATLAB:  A = reshape(0:5, 2, 3)   gives   0 2 4
                                          1 3 5
```
Predict what NumPy gives for the same numbers and the same shape.

In [ ]:
import numpy as np

In [ ]:
# Same numbers, same shape, different cells
A = np.arange(6).reshape(2, 3)
print("reshape(2, 3):\n", A)
print("flattened:", A.ravel())

In [ ]:
# The MATLAB layout, when porting code or reading MATLAB's raw files
A = np.arange(6).reshape(2, 3, order="F")
print('reshape(2, 3, order="F"):\n', A)
print('flattened with order="F":', A.ravel(order="F"))

In [ ]:
# What it costs: a recording of 2 channels, stored sample by sample
# (channel 1 holds 10 11 12, channel 2 holds 20 21 22)
recording = np.array([10, 20, 11, 21, 12, 22])
# MATLAB: reshape(recording, 2, 3) puts one channel per row.
print("the same call, ported as it is:\n", recording.reshape(2, 3))
print('with order="F":\n', recording.reshape(2, 3, order="F"))

### Trap 12 - Logic on arrays needs & and parentheses.

`traps/12_boolean_logic.py`

Which lines work, which raise an error, and which give a wrong answer silently?

In [ ]:
import numpy as np

x = np.array([1, 5, 9])

In [ ]:
# Element by element, with parentheses: works
print("(x > 2) & (x < 8):", (x > 2) & (x < 8))

In [ ]:
# and / or work on a single True or False, not on an array
try:
    print((x > 2) and (x < 8))
except ValueError as error:
    print("(x > 2) and (x < 8): ValueError:", error)

In [ ]:
# Without parentheses, 2 & x is evaluated first
try:
    print(x > 2 & x < 8)
except ValueError as error:
    print("x > 2 & x < 8:       ValueError:", error)

In [ ]:
# With plain numbers there is no error, only a wrong answer
a = 1
print("is 1 between 2 and 8?")
print("  a > 2 & a < 8:    ", a > 2 & a < 8)
print("  (a > 2) & (a < 8):", (a > 2) & (a < 8))
print("  2 < a < 8:        ", 2 < a < 8)

In [ ]:
# An array in an if has the same problem: say what you mean
try:
    if x > 2:
        print("never printed")
except ValueError as error:
    print("if x > 2: ValueError:", error)
print("(x > 2).any():", (x > 2).any(), "| (x > 2).all():", (x > 2).all())